# OP-FL v2 — 06d. Puissance du test verrouillé et amendement 1 de la spécification gelée

**Pourquoi.** La spécification gelée au 06c fixe comme critère principal C1 : R²_τ ≥ 0,25 **et** un gain en \|CITL\| moyen. Or les hôpitaux verrouillés sont petits (100 à 499 séjours, environ 7 décès par hôpital).
Le bruit d'estimation de chaque a_h y dépasse donc l'hétérogénéité réelle entre hôpitaux, et C1 échouerait souvent même si la méthode marche comme en développement (simulation préliminaire : puissance de 0,27 à 0,47).

**Ce que fait ce notebook**, sans lire aucun résultat du test verrouillé :
1. Il vérifie que la spécification gelée et les modules n'ont pas changé depuis le 06c (empreintes SHA-256).
2. Il estime sur les hôpitaux de développement la **pente de calibration au niveau des sites** b, avec les prédictions croisées de la spécification gelée (44 descripteurs).
3. Il calcule la **puissance** des critères amendés sur les **vrais effectifs** `n_val + n_test` des hôpitaux verrouillés. Il le fait avec **la procédure exacte de l'analyse finale** (module `opcal_amend`, que le notebook 07 réutilisera), et vérifie que l'erreur de type I est nominale.
4. Il écrit `amendement_1.json`, lié à l'empreinte de `specification_gelee.json` (qui n'est pas modifié), et le texte de dépôt `depot_osf.md`.

---

### Critères amendés (fixés ici, avant les résultats du test verrouillé)

Tous les tests sont **unilatéraux au seuil 2,5 %** (équivalent à un IC bilatéral à 95 %).

**Estimand principal.** b dans a_h = α + b·(â_h − moyenne de â) + u_h + e_h, avec u_h ~ N(0, τ_r²) et e_h ~ N(0, v_h).
* a_h : CITL du modèle final `x_only` sur les patients `val + test` de l'hôpital verrouillé h.
* â_h : prédiction OP-Cal gelée, traitée comme une covariable fixe.
* v_h = 1 / Σ μ(1 − μ).

**Interprétation de b.** À moyenne correcte, l'erreur quadratique de calibration d'OP-Cal est inférieure à celle de la meilleure constante de (2b − 1)·Var(â). Donc :
* b > 0 : le signal se transporte ;
* b > 1/2 : OP-Cal améliore la calibration ;
* b = 1 : les prédictions sont à la bonne échelle.

| Code | Famille | Hypothèse (H1) | Statistique et test |
|---|---|---|---|
| **C1a** | 1 | b > 0 | REML, Knapp-Hartung (t, k − 2 ddl) |
| **C4** | 1 | γ > 0 | hyperlactatémie : logistique avec offset du score, γ sur log(taux de dosage) centré ; bootstrap de 2000 hôpitaux |
| **C1b** | 2 | b > 0,5 | comme C1a |
| **C2** | 2 | Δℓ > 0 | gain de log-vraisemblance par patient, OP-Cal contre E0 (aucune correction) ; bootstrap de 2000 hôpitaux |

**Multiplicité (gatekeeping séquentiel).** La famille 1 est testée par Holm : le plus petit p doit être ≤ 1,25 % et le second ≤ 2,5 %. La famille 2 n'est testée que si **toute** la famille 1 est rejetée, en séquence fixe : C1b, puis C2, chacun à 2,5 %.

**Toujours rapportés.** α, b et R²_τ avec leurs IC ; bootstrap de b (sensibilité) ; gains en \|CITL\| et en log-vraisemblance contre E0 et E1 ; EM et BBSE ; GRU-D ; S_mech_fixe ; C3 (k labels) en descriptif.

**Formulation des conclusions.**
* C1a seul : « le signal se transporte, l'ampleur du gain reste incertaine ».
* C1b : « amélioration confirmée ».
* C2 : « gain de vraisemblance confirmé contre l'absence de correction ».
* Un échec est rapporté tel quel.

## Étape 0 — Configuration, empreintes et statut de validation

`VALIDATION` reste à `False` tant que l'amendement n'a pas été relu par l'encadrante. Une fois validé :
1. remplir `par` et `date` ;
2. mettre `valide` à `True` ;
3. ré-exécuter tout le notebook.

`amendement_1.json` passe alors au statut « validé ». C'est cette version, avec son empreinte, qu'il faut déposer.

In [1]:
# ============================================================
# ÉTAPE 0 — CONFIGURATION, EMPREINTES ET STATUT DE VALIDATION
# ============================================================
import sys, json, importlib, warnings, time, re, hashlib
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
warnings.filterwarnings("ignore", category=RuntimeWarning)

VALIDATION = {"valide": False, "par": "", "date": ""}       # à remplir après relecture par l'encadrante

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data_v1"
SRC_DIR = PROJECT_DIR / "src"
RES02 = PROJECT_DIR / "results" / "02_baselines"
RES04 = PROJECT_DIR / "results" / "04_observation_label"
RES05 = PROJECT_DIR / "results" / "05_raccourci_masquage"
RES06B = PROJECT_DIR / "results" / "06b_faisabilite_opcal"
RES06C = PROJECT_DIR / "results" / "06c_robustesse_opcal"
RES_DIR = PROJECT_DIR / "results" / "06d_amendement_puissance"
RES_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR.resolve()))

EXPECTED_PROTOCOL_HASH = "d92099ea7c11d517"
protocol = json.loads((DATA_DIR / "splits" / "protocol.json").read_text(encoding="utf-8"))
assert protocol["hash"] == EXPECTED_PROTOCOL_HASH, f"Protocole inattendu ({protocol['hash']})."
FOLDS = sorted(int(k) for k in protocol["folds"])
FOLD_OF = {int(h): int(k) for k in protocol["folds"] for h in protocol["folds"][k]}
DEV = sorted(FOLD_OF)
LOCKED = set(int(h) for h in protocol["locked_test_hospitals"])
STAY_LABELS = RES04 / "stay_labels.parquet"

def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

SPEC_PATH = RES06C / "specification_gelee.json"
assert SPEC_PATH.exists(), "Exécuter d'abord le notebook 06c."
SPEC = json.loads(SPEC_PATH.read_text(encoding="utf-8"))
assert SPEC["statut"] == "gelée", f"Spécification au statut « {SPEC['statut']} » : pas d'amendement possible, revoir le 06c."
SPEC_SHA = sha256(SPEC_PATH)
m = re.search(r"SHA-256 `([0-9a-f]{16})", (RES06C / "rapport_06c.md").read_text(encoding="utf-8"))
SPEC_SHA_06C = m.group(1) if m else None
MOD_SHA = {f: sha256(SRC_DIR / f) for f in ("opcal_feas.py", "opcal_robust.py")}

N_REP_M, N_BOOT_C2 = 500, 400          # simulations de mortalité (C1a, C1b, C2)
N_REP_G, N_BOOT_G = 150, 300           # simulations du lactate (C4)
N_PART, N_BOOT, SEED = 50, 2000, 2026
print(f"Protocole {protocol['hash']} | spécification « {SPEC['statut']} » du {SPEC['date']} | SHA-256 {SPEC_SHA[:16]}… "
      f"(06c : {SPEC_SHA_06C}…)")
print("Modules :", {k: v[:12] for k, v in MOD_SHA.items()}, "| validation :", VALIDATION)

Protocole d92099ea7c11d517 | spécification « gelée » du 2026-10-26T18:32:44 | SHA-256 302a9b448a1adcf4… (06c : 302a9b448a1adcf4…)
Modules : {'opcal_feas.py': '9770a8614e75', 'opcal_robust.py': 'eaa1313c3521'} | validation : {'valide': False, 'par': '', 'date': ''}


## Étape 1 — Module `src/opcal_amend.py`

La procédure confirmatoire (C1a, C1b, C2, C4, Holm et gatekeeping) et la simulation de puissance, qui appelle exactement les mêmes fonctions.

In [2]:
%%writefile ../src/opcal_amend.py
"""
opcal_amend — procédure confirmatoire de l'amendement 1 et sa puissance (OP-Cal).

Généré par notebooks/06d_amendement_puissance.ipynb (étape 1) : modifier le notebook, pas ce fichier.
Les fonctions de test sont celles que le notebook 07 appliquera au test verrouillé ; la simulation de
puissance les appelle telles quelles, pour que puissance et analyse finale suivent la même procédure.

Tous les tests sont unilatéraux au seuil 2,5 % (équivalent à un IC bilatéral à 95 %).
  C1a : b > 0      (le signal de calibration des sites se transporte)
  C1b : b > 0,5    (amélioration : à moyenne correcte, l'erreur quadratique de calibration baisse de
                    (2b − 1)·Var(â) par rapport à la meilleure constante)
  C4  : γ > 0      (mécanisme Y* = D × Y : effet du log du taux de dosage, à score égal)
  C2  : Δℓ > 0     (gain de log-vraisemblance par patient d'OP-Cal par rapport à E0, aucune correction)
Famille 1 = {C1a, C4} (Holm) ; famille 2 = C1b puis C2 (séquence fixe), testée seulement si toute la
famille 1 est rejetée (gatekeeping séquentiel : risque global d'erreur contrôlé à 2,5 % unilatéral).
"""
import numpy as np
import pandas as pd
from scipy import optimize, stats

import opcal_feas as oc

ALPHA = 0.025   # unilatéral


# ----------------------------------------------------------------------------------------------
# C1 : méta-régression REML de a_h sur la prédiction gelée, IC de Knapp-Hartung
# ----------------------------------------------------------------------------------------------

def reml_metareg(y, x, v):
    """a_h = alpha + b·x_h + u_h + e_h, u_h ~ N(0, tau2), e_h ~ N(0, v_h) ; x_h = â_h − moyenne(â).
    tau2 par REML ; variance de b par Knapp-Hartung (sans troncature), t à k − 2 ddl."""
    y, x, v = (np.asarray(a, float) for a in (y, x, v))
    X = np.column_stack([np.ones(len(y)), x])

    def fit(t2):
        w = 1.0 / (v + t2)
        A = X.T @ (X * w[:, None])
        beta = np.linalg.solve(A, (X * w[:, None]).T @ y)
        return w, A, beta, y - X @ beta

    def nll(lt2):
        t2 = np.exp(lt2)
        w, A, beta, r = fit(t2)
        return 0.5 * (np.sum(np.log(v + t2)) + np.linalg.slogdet(A)[1] + np.sum(w * r ** 2))

    lt2 = optimize.minimize_scalar(nll, bounds=(np.log(1e-8), np.log(10.0)), method="bounded",
                                   options={"xatol": 1e-6}).x
    t2 = float(np.exp(lt2))
    w, A, beta, r = fit(t2)
    df = len(y) - 2
    s2 = float(np.sum(w * r ** 2) / df)
    se = float(np.sqrt(s2 * np.linalg.inv(A)[1, 1]))
    return {"alpha": float(beta[0]), "b": float(beta[1]), "se_b": se, "df": df, "tau2_resid": t2, "k": len(y)}


def p_slope(res, b0):
    """p unilatéral pour H0 : b ≤ b0 (Knapp-Hartung)."""
    return float(stats.t.sf((res["b"] - b0) / res["se_b"], res["df"]))


def ci_slope(res, level=0.95):
    q = stats.t.ppf(0.5 + level / 2, res["df"])
    return res["b"] - q * res["se_b"], res["b"] + q * res["se_b"]


def boot_slope(y, x, v, n_boot=2000, seed=0):
    """Sensibilité : bootstrap des hôpitaux, tau2 réestimé à chaque tirage."""
    y, x, v = (np.asarray(a, float) for a in (y, x, v))
    rng = np.random.default_rng(seed)
    out = np.empty(n_boot)
    for i in range(n_boot):
        j = rng.integers(0, len(y), len(y))
        out[i] = reml_metareg(y[j], x[j] - x[j].mean(), v[j])["b"]
    return out


def c1_analysis(cal, ahat):
    """cal : citl, var_citl par hôpital ; ahat : prédiction gelée (même index)."""
    a = ahat.reindex(cal.index).to_numpy(float)
    res = reml_metareg(cal["citl"].to_numpy(), a - a.mean(), cal["var_citl"].to_numpy())
    lo, hi = ci_slope(res)
    raw = oc.random_effects(cal["citl"], cal["var_citl"])
    resid = oc.random_effects(cal["citl"].to_numpy() - a, cal["var_citl"].to_numpy())
    return {**res, "b_IC_bas": lo, "b_IC_haut": hi, "p_C1a": p_slope(res, 0.0), "p_C1b": p_slope(res, 0.5),
            "var_ahat": float(np.var(a)), "R2_tau": float(1 - resid["tau2"] / raw["tau2"]) if raw["tau2"] > 0 else np.nan}


# ----------------------------------------------------------------------------------------------
# C2 : gain de log-vraisemblance par patient (OP-Cal contre E0), bootstrap des hôpitaux
# ----------------------------------------------------------------------------------------------

def loglik(lp, y):
    p = np.clip(oc.sigmoid(lp), 1e-12, 1 - 1e-12)
    return float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))


def loglik_gain_by_hospital(stays, ahat, base=None):
    """Δℓ_h = ℓ(lp + â_h) − ℓ(lp + base_h) sur les patients d'évaluation ; base = 0 (E0) par défaut."""
    rows = []
    for h, g in stays.groupby("hospitalid"):
        lp, y = g["lp"].to_numpy(), g["y"].to_numpy(float)
        b0 = 0.0 if base is None else float(base.loc[h])
        rows.append({"hospitalid": h, "gain": loglik(lp + float(ahat.loc[h]), y) - loglik(lp + b0, y), "n": len(g)})
    return pd.DataFrame(rows).set_index("hospitalid")


def ratio_boot(num, den, n_boot=2000, seed=0):
    """Σ num / Σ den, IC percentile et p unilatéral (H0 : ≤ 0) par bootstrap des hôpitaux."""
    num, den = np.asarray(num, float), np.asarray(den, float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(num), (n_boot, len(num)))
    bs = num[idx].sum(1) / den[idx].sum(1)
    est = float(num.sum() / den.sum())
    return est, float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975)), float((1 + np.sum(bs <= 0)) / (n_boot + 1))


# ----------------------------------------------------------------------------------------------
# C4 : γ (log du taux de dosage), régression logistique avec offset, bootstrap des hôpitaux
# ----------------------------------------------------------------------------------------------

def gamma_test(stays, z_by_h, n_boot=2000, seed=0):
    """logit P(Y* = 1) = lp + alpha + γ·(z_h − moyenne) ; IC percentile et p unilatéral (H0 : γ ≤ 0)."""
    s = stays[stays["hospitalid"].isin(z_by_h.dropna().index)]
    z = s["hospitalid"].map(z_by_h).to_numpy(float)
    z = z - np.mean(z_by_h.dropna())
    g = s["hospitalid"].to_numpy()
    lp, y = s["lp"].to_numpy(), s["y"].to_numpy(float)
    est = oc.offset_slope_fit(lp, y, z)[1]
    hs = np.unique(g)
    rows = {h: np.flatnonzero(g == h) for h in hs}
    rng = np.random.default_rng(seed)
    bs = np.empty(n_boot)
    for b in range(n_boot):
        i = np.concatenate([rows[h] for h in rng.choice(hs, len(hs))])
        bs[b] = oc.offset_slope_fit(lp[i], y[i], z[i])[1]
    return {"gamma": float(est), "IC_bas": float(np.quantile(bs, 0.025)), "IC_haut": float(np.quantile(bs, 0.975)),
            "p_C4": float((1 + np.sum(bs <= 0)) / (n_boot + 1)), "IC_contient_1": bool(np.quantile(bs, 0.025) <= 1 <= np.quantile(bs, 0.975))}


# ----------------------------------------------------------------------------------------------
# Décision : Holm (famille 1), puis séquence fixe (famille 2)
# ----------------------------------------------------------------------------------------------

def decide(p_C1a, p_C4, p_C1b, p_C2, alpha=ALPHA):
    ps = sorted([("C1a", p_C1a), ("C4", p_C4)], key=lambda t: t[1])
    rej = {"C1a": False, "C4": False, "C1b": False, "C2": False}
    if ps[0][1] <= alpha / 2:
        rej[ps[0][0]] = True
        if ps[1][1] <= alpha:
            rej[ps[1][0]] = True
    if rej["C1a"] and rej["C4"]:
        if p_C1b <= alpha:
            rej["C1b"] = True
            if p_C2 <= alpha:
                rej["C2"] = True
    return rej


# ----------------------------------------------------------------------------------------------
# Simulation de puissance (mêmes fonctions que l'analyse finale)
# ----------------------------------------------------------------------------------------------

def _sample_lp(pools, n, rng):
    """lp d'un hôpital simulé : tirage avec remise dans les patients d'un hôpital de développement pris au hasard."""
    pool = pools[rng.integers(0, len(pools))]
    return pool[rng.integers(0, len(pool), n)]


def simulate_mortality(n_eval, lp_pools, b, tau2, var_ahat, n_rep=500, n_boot_c2=400, seed=0, k_lab=25, n0_fixed=10.0):
    """Réplications du test verrouillé : â_h ~ N(0, var_ahat) (dispersion des prédictions du développement),
    a_h = b·â_h + u_h avec Var(a_h) = tau2 (si b²·var_ahat ≤ tau2). Retourne p_C1a, p_C1b, p_C2, b estimé."""
    rng = np.random.default_rng(seed)
    K = len(n_eval)
    s2u = max(tau2 - b ** 2 * var_ahat, 1e-4)
    out = []
    for r in range(n_rep):
        ahat = rng.normal(0, np.sqrt(var_ahat), K)
        a = b * ahat + rng.normal(0, np.sqrt(s2u), K)
        c, v, gain = np.empty(K), np.empty(K), np.empty(K)
        reg = np.full((K, 3), np.nan)      # C3 tel que gelé : |CITL| déploiement (local, rétrécissement fixe, EB OP-Cal)
        for h in range(K):
            lp = _sample_lp(lp_pools, int(n_eval[h]), rng)
            y = (rng.random(len(lp)) < oc.sigmoid(lp + a[h])).astype(float)
            c[h], v[h] = oc.offset_fit(lp, y)
            gain[h] = loglik(lp + ahat[h], y) - loglik(lp, y)
            perm = rng.permutation(len(lp)); half = len(lp) // 2
            if half >= k_lab:
                L, Dp = perm[:half][:k_lab], perm[half:]
                res_ = lambda off: abs(oc.offset_fit(lp[Dp] + off, y[Dp])[0])
                reg[h] = (res_(oc.offset_fit(lp[L], y[L])[0]),
                          res_(oc.offset_fit(lp[L], y[L], prior_mean=0.0, prior_var=1.0 / n0_fixed)[0]),
                          res_(oc.offset_fit(lp[L], y[L], prior_mean=ahat[h], prior_var=s2u + 1e-3)[0]))
        res = reml_metareg(c, ahat - ahat.mean(), v)
        _, _, _, p2 = ratio_boot(gain, n_eval, n_boot_c2, int(rng.integers(1e9)))
        # C1 tel que gelé au 06c : R²_τ ≥ 0,25 et borne basse de l'IC du gain |CITL|(E1) − |CITL|(OP-Cal) > 0
        t_raw = oc.random_effects(c, v)["tau2"]
        r2 = 1 - oc.random_effects(c - ahat, v)["tau2"] / t_raw if t_raw > 0 else np.nan
        diff = np.abs(c) - np.abs(c - ahat)          # E1 = moyenne vraie, nulle dans la simulation
        bs = diff[rng.integers(0, K, (n_boot_c2, K))].mean(1)
        ok = np.isfinite(reg[:, 0])
        comp = 0 if np.nanmean(reg[:, 0]) <= np.nanmean(reg[:, 1]) else 1
        d3 = reg[ok, comp] - reg[ok, 2]
        c3 = bool(ok.sum() > 2 and np.quantile(d3[rng.integers(0, ok.sum(), (n_boot_c2, ok.sum()))].mean(1), 0.025) > 0)
        out.append({"rep": r, "b_est": res["b"], "p_C1a": p_slope(res, 0.0), "p_C1b": p_slope(res, 0.5), "p_C2": p2, "C3_gele": c3,
                    "C1_gele": bool(np.isfinite(r2) and r2 >= 0.25 and np.quantile(bs, 0.025) > 0), "R2_tau": r2,
                    "largeur_IC": 2 * stats.t.ppf(0.975, res["df"]) * res["se_b"]})
    return pd.DataFrame(out)


def simulate_gamma(n_eval48, lp_pools, z_pool, gamma, tau2_resid, n_rep=150, n_boot=300, seed=0):
    """Réplications de C4 : z_h tiré parmi les log taux de dosage du développement ; Y* ~ lp + γ·(z − z̄) + u_h."""
    rng = np.random.default_rng(seed)
    K = len(n_eval48)
    out = []
    for r in range(n_rep):
        z = rng.choice(z_pool, K)
        u = rng.normal(0, np.sqrt(max(tau2_resid, 1e-6)), K)
        parts = []
        for h in range(K):
            lp = _sample_lp(lp_pools, int(n_eval48[h]), rng)
            y = (rng.random(len(lp)) < oc.sigmoid(lp + gamma * (z[h] - z.mean()) + u[h])).astype(float)
            parts.append(pd.DataFrame({"hospitalid": h, "lp": lp, "y": y}))
        st = pd.concat(parts, ignore_index=True)
        t = gamma_test(st, pd.Series(z, index=range(K)), n_boot, int(rng.integers(1e9)))
        out.append({"rep": r, "gamma_est": t["gamma"], "p_C4": t["p_C4"], "evenements": int(st["y"].sum())})
    return pd.DataFrame(out)


def power_table(sim_m, sim_g, alpha=ALPHA, seed=0):
    """Puissance de chaque critère et de la procédure complète (réplications de mortalité appariées au hasard
    avec des réplications de C4, indépendantes)."""
    rng = np.random.default_rng(seed)
    pg = sim_g["p_C4"].to_numpy()[rng.integers(0, len(sim_g), len(sim_m))]
    dec = [decide(a, g, b, c, alpha) for a, g, b, c in zip(sim_m["p_C1a"], pg, sim_m["p_C1b"], sim_m["p_C2"])]
    return {"C1_gele": float(np.mean(sim_m["C1_gele"])), "C3_gele": float(np.mean(sim_m["C3_gele"])),
            "C1a_seul": float(np.mean(sim_m["p_C1a"] <= alpha)), "C1b_seul": float(np.mean(sim_m["p_C1b"] <= alpha)),
            "C2_seul": float(np.mean(sim_m["p_C2"] <= alpha)), "C4_seul": float(np.mean(sim_g["p_C4"] <= alpha)),
            "famille1": float(np.mean([d["C1a"] and d["C4"] for d in dec])),
            "C1b_confirme": float(np.mean([d["C1b"] for d in dec])), "C2_confirme": float(np.mean([d["C2"] for d in dec])),
            "b_estime_median": float(sim_m["b_est"].median()), "largeur_IC_mediane": float(sim_m["largeur_IC"].median()),
            "R2_tau_median": float(np.nanmedian(sim_m["R2_tau"]))}

Writing ../src/opcal_amend.py


## Étape 2 — Cibles et prédictions gelées sur les hôpitaux de développement

Mêmes cibles a_h qu'aux 06b et 06c. La prédiction croisée utilise **les 44 descripteurs de la spécification gelée**. Le contrôle de l'étape 6 vérifie qu'on retrouve le R²_τ de l'ablation du 06c.

In [3]:
# ============================================================
# ÉTAPE 2 — CIBLES ET PRÉDICTIONS GELÉES (DÉVELOPPEMENT)
# ============================================================
import opcal_feas as oc, opcal_robust as orb, opcal_amend as am
for m_ in (oc, orb, am):
    importlib.reload(m_)

TASKS = {"mortalite_x_only": {"pred_dir": RES02 / "preds", "name": "x_only", "label": "y_mortality", "pop": "tous"},
         "lactate_x_only": {"pred_dir": RES05 / "preds", "name": "A_lac_neg_x_only", "label": "hyper_neg", "pop": "present48"}}
st = pd.read_parquet(STAY_LABELS).set_index("stay_id")
POP = {"tous": st.index, "present48": st.index[st["present48"].astype(bool)]}
CAL, STAYS = {}, {}
for t, c_ in TASKS.items():
    d, _ = oc.load_transfer(c_["pred_dir"], c_["name"], FOLDS)
    if c_["label"] != "y_mortality" or "y" not in d.columns:
        d = d.drop(columns="y", errors="ignore")
        d = d[d["stay_id"].isin(POP[c_["pop"]])].copy()
        d["y"] = d["stay_id"].map(st[c_["label"]]).astype(float)
    d = d[d["y"].notna()].reset_index(drop=True)
    CAL[t], STAYS[t] = oc.hospital_calibration(d), d

D_SITE, GROUPS = oc.site_descriptors(DATA_DIR, STAY_LABELS, DEV)
D_SITE["log_test_rate_lactate"] = np.log(D_SITE["test_rate_lactate"].clip(lower=1e-3))
D_M = D_SITE.join(CAL["mortalite_x_only"][["mean_lp", "sd_lp"]])
FROZEN = SPEC["meta_modele"]["descripteurs"]
assert all(c in D_M.columns for c in FROZEN), "Descripteurs gelés introuvables."
cal = CAL["mortalite_x_only"]
PRIOR_DEV = oc.crossfit_prior(cal, D_M, FROZEN)
ev = oc.evaluate_prior(cal, PRIOR_DEV)
print(f"Mortalité x_only, {len(FROZEN)} descripteurs gelés, plis du protocole : R²_τ = {ev['R2_tau']:.3f} ; |CITL| = {ev['mean_abs_citl']:.3f}")

Mortalité x_only, 44 descripteurs gelés, plis du protocole : R²_τ = 0.608 ; |CITL| = 0.244


## Étape 3 — Pente de calibration au niveau des sites, sur le développement

Elle est mesurée avec les mêmes fonctions que C1a et C1b, sur les prédictions croisées : plis du protocole, 50 partitions aléatoires, leave-one-hospital-out.
Avec environ 24 décès par hôpital, cette estimation est plus précise qu'elle ne le sera sur le test verrouillé.
Si b y est nettement inférieur à 1, les prédictions sont trop dispersées, et C1b sera difficile à confirmer.

In [4]:
# ============================================================
# ÉTAPE 3 — PENTE b AU NIVEAU DES SITES (DÉVELOPPEMENT)
# ============================================================
rows = []
def slope_row(name, prior):
    r = am.c1_analysis(cal, prior["a_hat"])
    rows.append({"schéma": name, "b": r["b"], "b_IC_bas": r["b_IC_bas"], "b_IC_haut": r["b_IC_haut"], "alpha": r["alpha"],
                 "p_C1a": r["p_C1a"], "p_C1b": r["p_C1b"], "R2_tau": r["R2_tau"], "var_ahat": r["var_ahat"]})
    return r
r_proto = slope_row("plis du protocole", PRIOR_DEV)
rng = np.random.default_rng(SEED)
part = []
for i in range(N_PART):
    f = np.empty(len(cal), int)
    f[rng.permutation(len(cal))] = np.arange(len(cal)) % len(FOLDS)
    c2 = cal.copy(); c2["fold"] = f
    r = am.c1_analysis(cal, oc.crossfit_prior(c2, D_M, FROZEN)["a_hat"])
    part.append(r)
PART = pd.DataFrame(part)
_, pr_loho = orb.leave_one_hospital_out(cal, D_M, FROZEN)
r_loho = slope_row("leave-one-hospital-out", pr_loho)
rows.append({"schéma": f"{N_PART} partitions (médiane)", "b": PART["b"].median(), "b_IC_bas": PART["b"].quantile(0.05),
             "b_IC_haut": PART["b"].quantile(0.95), "alpha": PART["alpha"].median(), "p_C1a": PART["p_C1a"].median(),
             "p_C1b": PART["p_C1b"].median(), "R2_tau": PART["R2_tau"].median(), "var_ahat": PART["var_ahat"].median()})
TAB_DEV = pd.DataFrame(rows).set_index("schéma")
TAB_DEV.to_csv(RES_DIR / "pente_developpement.csv")
print("Pente de calibration des sites (développement) ; pour les partitions, les bornes sont les 5e et 95e centiles :")
display(TAB_DEV.round(3))

bs = am.boot_slope(cal["citl"], PRIOR_DEV["a_hat"].reindex(cal.index) - PRIOR_DEV["a_hat"].mean(), cal["var_citl"], 1000, SEED)
print(f"Bootstrap des hôpitaux (plis du protocole) : b = {r_proto['b']:.3f} [{np.quantile(bs, 0.025):.3f} ; {np.quantile(bs, 0.975):.3f}]")

# gains de log-vraisemblance sur le développement (descriptif) : OP-Cal contre E0 et contre E1
E1_DEV = oc.crossfit_prior(cal, D_M, [])["a_hat"]
g0 = am.loglik_gain_by_hospital(STAYS["mortalite_x_only"], PRIOR_DEV["a_hat"])
g1 = am.loglik_gain_by_hospital(STAYS["mortalite_x_only"], PRIOR_DEV["a_hat"], base=E1_DEV)
GAIN_DEV = pd.DataFrame([dict(zip(["gain_par_patient", "IC_bas", "IC_haut", "p_unilateral"], am.ratio_boot(g["gain"], g["n"], N_BOOT, SEED)), contre=k)
                         for k, g in (("E0", g0), ("E1", g1))]).set_index("contre")
print("Gain de log-vraisemblance par patient (développement, prédictions croisées) :")
display(GAIN_DEV.round(5))

B_DEV, VAR_AHAT, TAU2_DEV = float(PART["b"].median()), float(PART["var_ahat"].median()), oc.random_effects(cal["citl"], cal["var_citl"])["tau2"]
print(f"Paramètres retenus pour la simulation : b = {B_DEV:.3f} (médiane des partitions) ; Var(â) = {VAR_AHAT:.4f} ; τ² brut = {TAU2_DEV:.4f}")

Pente de calibration des sites (développement) ; pour les partitions, les bornes sont les 5e et 95e centiles :


,b,b_IC_bas,b_IC_haut,alpha,p_C1a,p_C1b,R2_tau,var_ahat
schéma,,,,,,,,
plis du protocole,1.162,0.772,1.552,0.055,0.0,0.001,0.608,0.042
leave-one-hospital-out,0.968,0.628,1.309,0.057,0.0,0.004,0.556,0.057
50 partitions (médiane),0.916,0.756,1.085,0.058,0.0,0.024,0.399,0.047


Bootstrap des hôpitaux (plis du protocole) : b = 1.162 [0.815 ; 1.467]
Gain de log-vraisemblance par patient (développement, prédictions croisées) :


,gain_par_patient,IC_bas,IC_haut,p_unilateral
contre,,,,
E0,0.00162,0.00054,0.00286,0.0010
E1,0.00154,0.00060,0.00271,0.0005


Paramètres retenus pour la simulation : b = 0.916 (médiane des partitions) ; Var(â) = 0.0470 ; τ² brut = 0.0812


## Étape 4 — Puissance sur les vrais effectifs du test verrouillé

Seuls `n_val` et `n_test` des hôpitaux verrouillés sont lus dans le manifest (déjà lus au 06c pour fixer k).
Les scores lp des patients simulés sont tirés, sans leurs labels, parmi ceux des patients d'évaluation des hôpitaux de développement : la distribution du risque et le case-mix sont donc réalistes.
Les prédictions â_h ont la dispersion observée sur le développement.

* **Scénarios nuls** (b = 0 pour C1a, b = 0,5 pour C1b, γ = 0 pour C4) : la proportion de rejets doit rester proche de 2,5 %.
* **Scénarios alternatifs** : b = 0,8, b = 1 et b du développement ; τ² du développement et τ² = 0,12 ; γ du développement et γ = 0,5.

Toute la procédure (Holm, puis gatekeeping) est appliquée à chaque réplication. Les réplications de mortalité et de lactate sont indépendantes et appariées au hasard.

In [5]:
# ============================================================
# ÉTAPE 4 — PUISSANCE SUR LES VRAIS EFFECTIFS DU TEST VERROUILLÉ
# ============================================================
man_cols = ["hospitalid", "role", "n_val", "n_test"]
man = pd.read_csv(DATA_DIR / "clients" / "manifest.csv", usecols=man_cols)
man_locked = man[man["hospitalid"].isin(LOCKED)]
N_EVAL = (man_locked["n_val"] + man_locked["n_test"]).to_numpy()
ev_dev = st[st["hospitalid"].isin(DEV) & st["split"].isin(["val", "test"])]
SHARE48 = float(ev_dev["present48"].astype(bool).mean())
N_EVAL48 = np.maximum((N_EVAL * SHARE48).round().astype(int), 5)
MORT_DEV = float(STAYS["mortalite_x_only"]["y"].mean())
EXP_DEATHS = float(np.median(N_EVAL) * MORT_DEV)
print(f"Test verrouillé : {len(N_EVAL)} hôpitaux ; patients d'évaluation médiane {np.median(N_EVAL):.0f} "
      f"[{N_EVAL.min()} ; {N_EVAL.max()}] ; présents à 48 h (part du développement {SHARE48:.2f}) médiane {np.median(N_EVAL48):.0f} ; "
      f"décès attendus par hôpital (médiane, mortalité du développement {MORT_DEV:.3f}) ≈ {EXP_DEATHS:.1f}")

K_CONF = int(SPEC["k_confirmatoire"])
V_MED = 1.0 / (np.median(N_EVAL) * MORT_DEV * (1 - MORT_DEV))
LP_M = [g["lp"].to_numpy() for _, g in STAYS["mortalite_x_only"].groupby("hospitalid")]
LP_L = [g["lp"].to_numpy() for _, g in STAYS["lactate_x_only"].groupby("hospitalid")]
Z_POOL = D_SITE["log_test_rate_lactate"].dropna().to_numpy()
cal_l = CAL["lactate_x_only"]
pr_mech = oc.crossfit_prior(cal_l, D_SITE, [], fixed_beta=("log_test_rate_lactate", 1.0))
TAU2_MECH = oc.random_effects(cal_l["citl"].to_numpy() - pr_mech["a_hat"].reindex(cal_l.index).to_numpy(), cal_l["var_citl"].to_numpy())["tau2"]
G_DEV = am.gamma_test(STAYS["lactate_x_only"], D_SITE["log_test_rate_lactate"], N_BOOT, SEED)
print(f"Lactate (développement) : γ = {G_DEV['gamma']:.3f} [{G_DEV['IC_bas']:.3f} ; {G_DEV['IC_haut']:.3f}] ; τ² résiduel (γ = 1) = {TAU2_MECH:.3f}")

t0 = time.time()
SIM_G = {name: am.simulate_gamma(N_EVAL48, LP_L, Z_POOL, g, TAU2_MECH, N_REP_G, N_BOOT_G, SEED + i)
         for i, (name, g) in enumerate([("γ = 0 (nul)", 0.0), ("γ = 0,5", 0.5), ("γ du dév.", G_DEV["gamma"])])}
print(f"Simulations de C4 : {time.time() - t0:.0f} s")
SCEN = [("nul : b = 0", 0.0, TAU2_DEV), ("frontière C1b : b = 0,5", 0.5, TAU2_DEV), ("b = 0,8", 0.8, TAU2_DEV),
        ("b = 1", 1.0, TAU2_DEV), ("b du dév.", B_DEV, TAU2_DEV), ("b = 0,8 ; τ² = 0,12", 0.8, 0.12),
        ("b = 1 ; τ² = 0,12", 1.0, 0.12), ("b du dév. ; τ² = 0,12", B_DEV, 0.12)]
rows, t0 = [], time.time()
for i, (name, b, tau2) in enumerate(SCEN):
    sim = am.simulate_mortality(N_EVAL, LP_M, b, tau2, VAR_AHAT, N_REP_M, N_BOOT_C2, SEED + 100 + i, k_lab=K_CONF)
    g_name = "γ = 0 (nul)" if b == 0 else "γ du dév."
    rows.append({"scénario": name, "b": b, "tau2": tau2, "C4 simulé avec": g_name, "contrainte_tau2": b ** 2 * VAR_AHAT > tau2,
                 **am.power_table(sim, SIM_G[g_name], seed=SEED + i)})
TAB_POW = pd.DataFrame(rows).set_index("scénario")
TAB_POW.to_csv(RES_DIR / "puissance.csv")
print(f"Simulations de mortalité : {time.time() - t0:.0f} s ({N_REP_M} réplications par scénario)")
display(TAB_POW.drop(columns=["contrainte_tau2"]).round(3))
TAB_G = pd.DataFrame({k: {"C4 (p ≤ 2,5 %)": float(np.mean(v["p_C4"] <= am.ALPHA)), "γ estimé médian": float(v["gamma_est"].median()),
                          "événements médians": float(v["evenements"].median())} for k, v in SIM_G.items()}).T
display(TAB_G.round(3))

# erreur de type I (doit rester proche de 2,5 % ; 2 erreurs-types de Monte Carlo)
MC = lambda n: 2 * np.sqrt(am.ALPHA * (1 - am.ALPHA) / n)
TYPE1 = {"C1a sous b = 0": TAB_POW.loc["nul : b = 0", "C1a_seul"], "C1b sous b = 0,5": TAB_POW.loc["frontière C1b : b = 0,5", "C1b_seul"],
         "C4 sous γ = 0": TAB_G.loc["γ = 0 (nul)", "C4 (p ≤ 2,5 %)"]}
print("Erreur de type I :", {k: round(v, 3) for k, v in TYPE1.items()}, f"(cible 0,025 ; marge Monte Carlo ≈ {MC(N_REP_M):.3f} / {MC(N_REP_G):.3f})")

Test verrouillé : 96 hôpitaux ; patients d'évaluation médiane 70 [30 ; 149] ; présents à 48 h (part du développement 0.58) médiane 40 ; décès attendus par hôpital (médiane, mortalité du développement 0.094) ≈ 6.5
Lactate (développement) : γ = 0.928 [0.516 ; 1.346] ; τ² résiduel (γ = 1) = 0.395
Simulations de C4 : 245 s
Simulations de mortalité : 459 s (500 réplications par scénario)


,b,tau2,C4 simulé avec,C1_gele,C3_gele,C1a_seul,C1b_seul,C2_seul,C4_seul,famille1,C1b_confirme,C2_confirme,b_estime_median,largeur_IC_mediane,R2_tau_median
scénario,,,,,,,,,,,,,,,
nul : b = 0,0.000,0.081,γ = 0 (nul),0.000,0.006,0.024,0.000,0.000,0.047,0.002,0.000,0.000,0.008,0.981,-0.844
"frontière C1b : b = 0,5",0.500,0.081,γ du dév.,0.020,0.018,0.432,0.016,0.018,0.987,0.426,0.016,0.010,0.445,0.979,-0.079
"b = 0,8",0.800,0.081,γ du dév.,0.200,0.104,0.874,0.186,0.222,0.987,0.854,0.184,0.162,0.741,0.937,0.407
b = 1,1.000,0.081,γ du dév.,0.398,0.230,0.972,0.498,0.562,0.987,0.962,0.492,0.448,0.956,0.895,0.718
b du dév.,0.916,0.081,γ du dév.,0.314,0.172,0.954,0.336,0.392,0.987,0.940,0.334,0.302,0.879,0.926,0.579
"b = 0,8 ; τ² = 0,12",0.800,0.120,γ du dév.,0.158,0.112,0.818,0.174,0.240,0.987,0.810,0.172,0.160,0.760,1.003,0.261
"b = 1 ; τ² = 0,12",1.000,0.120,γ du dév.,0.380,0.184,0.968,0.410,0.484,0.987,0.956,0.408,0.394,0.921,0.980,0.470
"b du dév. ; τ² = 0,12",0.916,0.120,γ du dév.,0.270,0.152,0.934,0.288,0.358,0.987,0.918,0.284,0.254,0.853,0.996,0.358


,"C4 (p ≤ 2,5 %)",γ estimé médian,événements médians
γ = 0 (nul),0.047,-0.024,103.5
"γ = 0,5",0.727,0.457,107.0
γ du dév.,0.987,0.934,118.0


Erreur de type I : {'C1a sous b = 0': np.float64(0.024), 'C1b sous b = 0,5': np.float64(0.016), 'C4 sous γ = 0': np.float64(0.047)} (cible 0,025 ; marge Monte Carlo ≈ 0.014 / 0.025)


## Étape 5 — Amendement 1 et texte de dépôt

`amendement_1.json` cite l'empreinte complète de `specification_gelee.json`, qui reste inchangé. `depot_osf.md` est le texte du dépôt, en anglais.
Il faut déposer les deux fichiers JSON et ce texte, par exemple sur OSF Registries ou dans un commit git horodaté, **avant** le notebook 07.

In [6]:
# ============================================================
# ÉTAPE 5 — AMENDEMENT 1 ET TEXTE DE DÉPÔT
# ============================================================
crit_old = {c["code"]: c for c in SPEC["criteres_confirmatoires"]}
p_main = TAB_POW.loc["b du dév."]
PW_GEL, PW_C3 = float(p_main["C1_gele"]), float(p_main["C3_gele"])
BRUIT_FR = (f"variance d'échantillonnage médiane de a_h ≈ {V_MED:.3f} (environ {EXP_DEATHS:.0f} décès attendus par hôpital verrouillé) "
            f"contre τ² ≈ {TAU2_DEV:.3f} entre hôpitaux de développement")
BRUIT_EN = (f"median sampling variance of a_h ≈ {V_MED:.3f} (about {EXP_DEATHS:.0f} expected deaths per locked hospital) versus "
            f"between-hospital τ² ≈ {TAU2_DEV:.3f} in development")
if PW_GEL < 0.80:
    MOTIF_FR = (f"Puissance insuffisante du C1 gelé : {PW_GEL:.2f} sous les estimations du développement ({BRUIT_FR}). "
                "Simulation sur les vrais effectifs, avec la procédure finale (tableau 'puissance'). L'amendement précise aussi l'estimand, "
                "le lien avec l'amélioration de la calibration (b > 1/2) et le contrôle de la multiplicité.")
    MOTIF_EN = (f"The frozen primary criterion had insufficient power ({PW_GEL:.2f} under the development estimates; {BRUIT_EN}). "
                "The amendment also defines the estimand, its link to calibration improvement (b > 1/2) and multiplicity control.")
else:
    MOTIF_FR = (f"Le C1 gelé a une puissance de {PW_GEL:.2f} sous les estimations du développement ({BRUIT_FR}), mais il combine deux "
                "estimations sans estimand défini, sans lien explicite avec l'amélioration de la calibration et sans contrôle de la "
                "multiplicité entre C1 à C4 ; l'amendement les précise (relecture méthodologique du 26/10/2026).")
    MOTIF_EN = (f"The frozen primary criterion had a power of {PW_GEL:.2f} under the development estimates ({BRUIT_EN}), but it combined two "
                "estimates without a defined estimand, an explicit link to calibration improvement, or multiplicity control across C1–C4; "
                "the amendment specifies them.")
AMEND = {
    "document": "Amendement 1 à la spécification gelée OP-Cal (06c)",
    "statut": "validé" if VALIDATION["valide"] else "proposé (à valider avant dépôt)",
    "validation": VALIDATION,
    "date": datetime.now().isoformat(timespec="seconds"),
    "protocol_hash": protocol["hash"],
    "specification": {"fichier": "results/06c_robustesse_opcal/specification_gelee.json", "sha256": SPEC_SHA,
                      "date": SPEC["date"], "modifiee": False},
    "acces_au_test_verrouille": "aucun label, aucune prédiction, aucun descripteur ; seuls n_val et n_test du manifest "
                                "(déjà lus au 06c pour fixer k)",
    "motif": MOTIF_FR,
    "modifications": [
        {"critere": "C1", "avant": f"{crit_old['C1']['mesure']} ; {crit_old['C1']['seuil']}",
         "apres": "remplacé par C1a (famille 1, principal) et C1b (famille 2) ; R²_τ et gains en |CITL| rapportés comme estimations"},
        {"critere": "C2", "avant": f"{crit_old['C2']['mesure']} ; {crit_old['C2']['seuil']}",
         "apres": "gain de log-vraisemblance par patient contre E0 (famille 2) ; |CITL| contre E0 et E1 descriptifs"},
        {"critere": "C3", "avant": f"{crit_old['C3']['mesure']} ; {crit_old['C3']['seuil']}",
         "apres": f"descriptif (puissance au test verrouillé {PW_C3:.2f} sous les estimations du développement ; preuve principale du régime 2 : 06b, F2)"},
        {"critere": "C4", "avant": f"{crit_old['C4']['mesure']} ; {crit_old['C4']['seuil']}",
         "apres": "inchangé sur le fond ; test et IC précisés ; on rapporte aussi si l'IC contient 1"},
        {"critere": "S1", "avant": f"{crit_old['S1']['mesure']} ; {crit_old['S1']['seuil']}", "apres": "GRU-D : C1a et C1b, descriptifs"},
    ],
    "estimand_principal": ("b dans a_h = alpha + b·(â_h − moyenne de â) + u_h + e_h, u_h ~ N(0, tau_r²), e_h ~ N(0, v_h). "
                           "À moyenne correcte, l'erreur quadratique de calibration baisse de (2b − 1)·Var(â) par rapport à la meilleure constante."),
    "definitions": {
        "a_h": "CITL (offset logistique, cible bornée à [0,5 ; n − 0,5]) du modèle final x_only sur les patients val + test de l'hôpital verrouillé h",
        "v_h": "1 / Σ μ(1 − μ), μ = sigmoid(lp + a_h)",
        "â_h": "prédiction de opcal_feas.SiteEB ajusté une fois sur les 60 hôpitaux de développement (descripteurs gelés), covariable fixe",
        "E0": "aucune correction du modèle final (décalage 0)",
        "E1": "décalage constant = moyenne REML des a_h croisés des hôpitaux de développement",
        "z_h": "log de la part des séjours présents à 48 h redosés en lactate après 24 h, split train de l'hôpital verrouillé",
        "population_C4": "séjours présents à 48 h, patients val + test des hôpitaux verrouillés, label hyperlactatémie (non dosé = négatif) du notebook 04",
    },
    "criteres": [
        {"code": "C1a", "famille": 1, "H1": "b > 0", "test": "REML, Knapp-Hartung sans troncature, t à k − 2 ddl, unilatéral 2,5 %"},
        {"code": "C4", "famille": 1, "H1": "γ > 0",
         "test": "logistique avec offset lp, γ sur z_h centré ; bootstrap de 2000 hôpitaux (graine 2026) ; p = (1 + #{γ* ≤ 0}) / 2001"},
        {"code": "C1b", "famille": 2, "H1": "b > 0,5", "test": "comme C1a, H0 : b ≤ 0,5"},
        {"code": "C2", "famille": 2, "H1": "Δℓ > 0",
         "test": "Δℓ = Σ_h Σ_i [ℓ(lp_i + â_h) − ℓ(lp_i)] / Σ_h n_h ; bootstrap de 2000 hôpitaux (graine 2026) ; p = (1 + #{Δℓ* ≤ 0}) / 2001"},
    ],
    "multiplicite": ("famille 1 = {C1a, C4} par Holm (plus petit p ≤ 0,0125, second ≤ 0,025) ; famille 2 testée seulement si la famille 1 "
                     "est entièrement rejetée, en séquence fixe C1b puis C2, chacun à 0,025 (gatekeeping séquentiel, unilatéral 2,5 %)"),
    "regles": ["aucun hôpital verrouillé exclu",
               "modèles finaux : réglages du notebook 02, entraînés sur les 60 hôpitaux de développement, prédictions moyennées sur les seeds",
               "indépendance : patients distincts entre hôpitaux ; â_h fixes ; erreur commune du modèle final absorbée par alpha",
               "le test verrouillé est évalué une seule fois ; tout écart est déclaré comme analyse exploratoire"],
    "toujours_rapportes": ["alpha, b et R²_τ avec IC", "bootstrap de b (2000 hôpitaux, tau² réestimé)",
                           "|CITL| et log-vraisemblance contre E0 et E1", "EM et BBSE", "GRU-D", "S_mech_fixe (lactate)",
                           "C3 (k labels) descriptif"],
    "formulation_des_conclusions": {"C1a seul": "le signal de calibration des sites se transporte ; l'ampleur du gain reste incertaine",
                                    "C1b": "amélioration de la calibration confirmée", "C2": "gain de vraisemblance confirmé contre l'absence de correction",
                                    "échec": "rapporté tel quel"},
    "pente_developpement": TAB_DEV.reset_index().to_dict(orient="records"),
    "puissance": TAB_POW.reset_index().to_dict(orient="records"),
    "puissance_C4": TAB_G.reset_index().rename(columns={"index": "scénario"}).to_dict(orient="records"),
    "erreur_type_I": TYPE1,
    "code_sha256": {**MOD_SHA, "opcal_amend.py": sha256(SRC_DIR / "opcal_amend.py")},
}
amend_path = RES_DIR / "amendement_1.json"
amend_path.write_text(json.dumps(AMEND, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
AMEND_SHA = sha256(amend_path)

osf = f"""# OP-Cal — confirmatory analysis plan for the eICU locked test set (frozen specification + amendment 1)

**Status:** {"validated" if VALIDATION["valide"] else "PROPOSED — not yet validated"}{(" by " + VALIDATION["par"] + " on " + VALIDATION["date"]) if VALIDATION["valide"] else ""}.
**Protocol hash:** `{protocol['hash']}` ({len(DEV)} development hospitals in {len(FOLDS)} leave-hospitals-out folds; {len(LOCKED)} locked hospitals with 100–499 stays by protocol, never accessed).

## Files and fingerprints
| File | SHA-256 |
|---|---|
| `specification_gelee.json` (frozen {SPEC['date']}) | `{SPEC_SHA}` |
| `amendement_1.json` ({AMEND['date']}) | `{AMEND_SHA}` |
| `opcal_feas.py` / `opcal_robust.py` / `opcal_amend.py` | `{AMEND['code_sha256']['opcal_feas.py'][:16]}…` / `{AMEND['code_sha256']['opcal_robust.py'][:16]}…` / `{AMEND['code_sha256']['opcal_amend.py'][:16]}…` |

## Method under test
OP-Cal predicts the calibration-in-the-large offset a_h of a federated model at a new hospital without any of its labels. It uses an
empirical-Bayes (REML) meta-regression on {len(FROZEN)} label-free site descriptors (observation process, case-mix, structure, score distribution),
fitted once on out-of-federation offsets of the 60 development hospitals. For labels defined by a test (hyperlactataemia, "not tested = negative"),
the theory Y* = D·Y predicts an offset slope γ = 1 on the log testing rate.

## Why the frozen primary criterion was amended (before any access to the locked set)
{MOTIF_EN} Power was simulated with the real locked-hospital sizes ({len(N_EVAL)} hospitals, median {np.median(N_EVAL):.0f} evaluation
patients) and the exact final procedure; all scenarios are in `amendement_1.json` (field `puissance`). The few-label criterion (k = {K_CONF})
becomes descriptive (power {PW_C3:.2f}).

## Confirmatory hypotheses (one-sided α = 2.5%, family-wise error controlled)
* **Family 1 (Holm):** C1a — site-level calibration slope b > 0 (REML meta-regression of locked a_h on frozen predictions, Knapp–Hartung);
  C4 — γ > 0 (offset logistic regression, hospital bootstrap).
* **Family 2 (only if family 1 is fully rejected; fixed sequence):** C1b — b > 0.5 (equivalent to a reduction of the expected squared
  calibration error versus the best constant); then C2 — per-patient log-likelihood gain versus no correction > 0.

Estimated power under the development estimates (b = {B_DEV:.2f}, γ = {G_DEV['gamma']:.2f}): C1a {p_main['C1a_seul']:.2f}; C4 {TAB_G.loc['γ du dév.', 'C4 (p ≤ 2,5 %)']:.2f}; family 1 {p_main['famille1']:.2f};
C1b confirmed {p_main['C1b_confirme']:.2f}; C2 confirmed {p_main['C2_confirme']:.2f}. Simulated type I error: C1a {TYPE1['C1a sous b = 0']:.3f},
C1b {TYPE1['C1b sous b = 0,5']:.3f}, C4 {TYPE1['C4 sous γ = 0']:.3f} (target 0.025).

## Always reported (descriptive)
α, b and R²_τ with CIs; bootstrap of b; |CITL| and log-likelihood versus no correction and versus a constant; label-shift estimators (EM,
BBSE); GRU-D; mechanism-only offset; few-label updating (k = {SPEC['k_confirmatoire']}).

## Reporting rules
C1a alone: "the site calibration signal transports; the size of the gain remains uncertain". C1b: "calibration improvement confirmed".
Failures are reported as such. The locked set is evaluated once.
"""
(RES_DIR / "depot_osf.md").write_text(osf, encoding="utf-8")
print(f"amendement_1.json : statut « {AMEND['statut']} » ; SHA-256 {AMEND_SHA}")
print(f"depot_osf.md écrit. Spécification gelée inchangée : SHA-256 {SPEC_SHA}")

amendement_1.json : statut « proposé (à valider avant dépôt) » ; SHA-256 a02997c2662a2fb725ea3baf4eee104afaabd6634e651cd4b20a62e36e268352
depot_osf.md écrit. Spécification gelée inchangée : SHA-256 302a9b448a1adcf481802ce6827e95e8c9afb623c3b7cf39ac3ca9f9eead25ae


## Étape 6 — Contrôles et rapport

In [7]:
# ============================================================
# ÉTAPE 6 — CONTRÔLES ET RAPPORT
# ============================================================
CONTROLES = []
oc.check("Protocole identique (hash)", protocol["hash"] == EXPECTED_PROTOCOL_HASH, CONTROLES)
oc.check("Spécification gelée inchangée depuis le 06c (empreinte du rapport 06c)", SPEC_SHA_06C is not None and SPEC_SHA.startswith(SPEC_SHA_06C), CONTROLES)
oc.check("Modules opcal_feas et opcal_robust identiques à ceux de la spécification", MOD_SHA == SPEC["code_sha256"], CONTROLES)
abl = pd.read_csv(RES06C / "ablation.csv").set_index(["tâche", "ensemble"])
r2_ref = abl.loc[("mortalite_x_only", "S_all − exclus (gelé)"), "R2_tau"]
oc.check(f"Prédiction gelée du développement reproduite (R²_τ {ev['R2_tau']:.4f} ; 06c {r2_ref:.4f})", abs(ev["R2_tau"] - r2_ref) < 1e-6, CONTROLES)
oc.check(f"Erreur de type I de C1a et C1b nominale (≤ 0,025 + {MC(N_REP_M):.3f})",
         TYPE1["C1a sous b = 0"] <= am.ALPHA + MC(N_REP_M) and TYPE1["C1b sous b = 0,5"] <= am.ALPHA + MC(N_REP_M), CONTROLES)
oc.check(f"Erreur de type I de C4 nominale (≤ 0,025 + {MC(N_REP_G):.3f})", TYPE1["C4 sous γ = 0"] <= am.ALPHA + MC(N_REP_G), CONTROLES)
oc.check("Test verrouillé : seuls hospitalid, role, n_val et n_test du manifest ont été lus", list(man_locked.columns) == man_cols, CONTROLES)
oc.check("Aucun hôpital verrouillé dans les cibles, descripteurs ou scores tirés pour la simulation",
         all(LOCKED.isdisjoint(STAYS[t]["hospitalid"]) and LOCKED.isdisjoint(CAL[t].index) for t in CAL) and LOCKED.isdisjoint(D_SITE.index), CONTROLES)
oc.check("Amendement lié à l'empreinte complète de la spécification", json.loads(amend_path.read_text(encoding="utf-8"))["specification"]["sha256"] == SPEC_SHA, CONTROLES)
CONTROLES = pd.DataFrame(CONTROLES)
print(f"\n{int(CONTROLES['ok'].sum())}/{len(CONTROLES)} contrôles réussis")

def md_table(df, digits=3):
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{v:.{digits}f}")
    esc = lambda x: str(x).replace("|", "\\|")
    head = "| " + " | ".join([esc(df.index.name or "")] + [esc(c) for c in df.columns]) + " |"
    sep = "|" + "---|" * (len(df.columns) + 1)
    body = ["| " + " | ".join([esc(i)] + [esc(v) for v in row]) + " |" for i, row in zip(df.index, df.to_numpy())]
    return "\n".join([head, sep] + body)

pow_cols = ["b", "tau2", "C1_gele", "C3_gele", "C1a_seul", "C4_seul", "famille1", "C1b_seul", "C1b_confirme", "C2_seul", "C2_confirme", "b_estime_median", "largeur_IC_mediane"]
rapport = f"""# Rapport 06d — puissance du test verrouillé et amendement 1

Généré le {datetime.now():%Y-%m-%d %H:%M} par `notebooks/06d_amendement_puissance.ipynb`. Protocole `{protocol['hash']}`. Agrégats uniquement.

**Amendement 1 : statut « {AMEND['statut']} »** — SHA-256 `{AMEND_SHA}`.
Spécification gelée inchangée — SHA-256 `{SPEC_SHA}`.

## Pente de calibration des sites (développement, {len(FROZEN)} descripteurs gelés)

{md_table(TAB_DEV)}

Gain de log-vraisemblance par patient (développement) :

{md_table(GAIN_DEV, 5)}

## Puissance (vrais effectifs du test verrouillé : {len(N_EVAL)} hôpitaux, médiane {np.median(N_EVAL):.0f} patients d'évaluation)

Paramètres : Var(â) = {VAR_AHAT:.4f} ; τ² du développement = {TAU2_DEV:.4f} ; variance d'échantillonnage médiane de a_h ≈ {V_MED:.3f} ;
k = {K_CONF} pour C3 ; {N_REP_M} réplications par scénario ({N_REP_G} pour C4).

**Motif de l'amendement :** {MOTIF_FR}

{md_table(TAB_POW[pow_cols])}

C4 (lactate) :

{md_table(TAB_G)}

Erreur de type I (cible 0,025) : {', '.join(f'{k} = {v:.3f}' for k, v in TYPE1.items())}.

## Contrôles

{md_table(CONTROLES.set_index('controle'))}
"""
(RES_DIR / "rapport_06d.md").write_text(rapport, encoding="utf-8")
print(f"Écrit : {RES_DIR / 'rapport_06d.md'} | amendement_1.json | depot_osf.md | puissance.csv | pente_developpement.csv")

OK     Protocole identique (hash)
OK     Spécification gelée inchangée depuis le 06c (empreinte du rapport 06c)
OK     Modules opcal_feas et opcal_robust identiques à ceux de la spécification
OK     Prédiction gelée du développement reproduite (R²_τ 0.6077 ; 06c 0.6077)
OK     Erreur de type I de C1a et C1b nominale (≤ 0,025 + 0.014)
OK     Erreur de type I de C4 nominale (≤ 0,025 + 0.025)
OK     Test verrouillé : seuls hospitalid, role, n_val et n_test du manifest ont été lus
OK     Aucun hôpital verrouillé dans les cibles, descripteurs ou scores tirés pour la simulation
OK     Amendement lié à l'empreinte complète de la spécification

9/9 contrôles réussis
Écrit : ..\results\06d_amendement_puissance\rapport_06d.md | amendement_1.json | depot_osf.md | puissance.csv | pente_developpement.csv
